In [9]:
# Imports
import os
import re
import json
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

In [10]:
# Get API key
load_dotenv(find_dotenv())
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
assert GOOGLE_API_KEY, "GOOGLE_API_KEY not found — check your .env at repo root"

In [11]:
# Initialize model
MODEL = "gemini-3.8-flash"
llm = ChatGoogleGenerativeAI(
    model=MODEL,
    temperature=0
)

In [12]:
# Helper functions to format the sources
def source_label(m):
    pages = f"p.{m['page_number']}" if m['page_number'] == m['end_page'] \
            else f"pp.{m['page_number']}-{m['end_page']}"
    return f"{m['title']} ({m['paper_id']}, {pages})"

def format_sources(docs):
    blocks = []
    for i, d in enumerate(docs, 1):
        blocks.append(f"[{i}] {source_label(d.metadata)}\n{d.page_content}")
    return "\n\n".join(blocks)

In [13]:
# Prep prompts and chain
SYSTEM = """You are a research assistant for an R&D team. Answer the question using ONLY the sources provided.
- Use only information found in the sources. If they don't contain the answer, say: "The provided sources do not contain this information."
- After every claim, cite the source it came from by its bracket number, e.g. [1].
- Do not use outside knowledge, and never invent a citation."""
HUMAN = "Question: {question}\n\nSources:\n{context}"

prompt = ChatPromptTemplate.from_messages([("system", SYSTEM), ("human", HUMAN)])
chain = prompt | llm

In [14]:
# Helper function to print citations
def show_answer(answer, docs):
    print(answer, "\n")
    cited = sorted(set(int(n) for n in re.findall(r"\[(\d+)\]", answer)))
    print("Sources:")
    for n in cited:
        if 1 <= n <= len(docs):
            print(f"  [{n}] {source_label(docs[n-1].metadata)}")
        else:
            print(f"  [{n}] ⚠️ cited source not in context")

In [15]:
# Get retrieved chunks

# DEV STAND-IN: pull a few chunks from chunks.json so the template is testable
# before Ivan's retrieval is wired in.
q = "What are the two RAG formulations introduced?"

with open(Path("../data/chunks.json"), encoding="utf-8") as f:
    raw = json.load(f)
docs = [Document(page_content=d["page_content"], metadata=d["metadata"])
        for d in raw[:5]]

# REAL RETRIEVAL (swap in when the store exists — delete the stand-in above):
# from langchain_chroma import Chroma
# store = Chroma(persist_directory="../data/chroma_db", embedding_function=EMB)  # EMB must match Tanish's
# docs = store.similarity_search(q, k=5)

In [16]:
# Create and call the chain
resp = chain.invoke({"question": q, "context": format_sources(docs)})
show_answer(resp.text, docs)

Based on the provided sources, the two RAG formulations introduced are:

1. One that conditions on the same retrieved passages across the whole generated sequence [2].
2. Another that can use different passages per token [2]. 

Sources:
  [2] Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks (2005.11401, p.1)
